In [ ]:
# from google.colab import drive
# drive.mount('/content/drive/')

# base_dir = "/content/drive/MyDrive/Blokus"
# import os
# # Change the current working directory to base_dir
# os.chdir(base_dir)

In [ ]:
from tqdm import tqdm
import torch
import numpy as np

from gymnasium_env import BlokusEnv, BlokusAction

from gymnasium_env.wrappers import MultipleColorsEncoding

from src.agents.qnetwork.training_algorithtms import TrainDQN, MyAlgo
from src.agents.qnetwork.archis import UnicolorArch, ColorfulArch, ColorfulArchEmbedding
from src.agents.qnetwork.qnetwork_agent import QNetworkAgent

from src.simulator import BlokusGameSimulator

from src.agents import RandomAgent, HeuristicAgent, MiniMaxAgent

from src.agents.heuristic.heuristics import greedy, min_his_expanders, max_my_expanders, min_his_possible_actions, min_actions_after_size, level_7

In [ ]:
BOARD_SIZE = 10
PLAYER_TURN = 1
DEVICE = (
    "cuda" if torch.cuda.is_available() else 
    "mps" if torch.backends.mps.is_available() else 
    "cpu"
)
USE_WANDB = False

In [ ]:
Random = RandomAgent(name="Random")
MinHisExpanders = HeuristicAgent(func=min_his_expanders, name="MinHisExpanders", board_size=BOARD_SIZE)
MaxMyExpanders = HeuristicAgent(func=max_my_expanders, name="MaxMyExpanders", board_size=BOARD_SIZE)
Greedy = HeuristicAgent(func=greedy, name="Greedy", board_size=BOARD_SIZE)
Level_10 = HeuristicAgent(func=level_7, name="Level_10", board_size=BOARD_SIZE)
MinHisActions = HeuristicAgent(func=min_his_possible_actions, name="MinHisActions", board_size=BOARD_SIZE)
MinActionsAfterSize = HeuristicAgent(func=min_actions_after_size, name="MinActionsAfterSize", board_size=BOARD_SIZE)

In [ ]:
agent = QNetworkAgent(
    board_size=BOARD_SIZE,
    model_class=ColorfulArchEmbedding,
    device=DEVICE,
    model_folder="final_my_approach"
)

wrappers = [MultipleColorsEncoding] if agent.model_class == ColorfulArchEmbedding or ColorfulArch else []

dummy_env = BlokusEnv(
    board_size=BOARD_SIZE,
    num_players=2,
)

for wrapper in wrappers:
    dummy_env = wrapper(dummy_env)
    
obs, info = dummy_env.reset()

trainer = MyAlgo(
    agent=agent,
    device=DEVICE,
    player_turn=PLAYER_TURN,
    wrappers=wrappers,
    batch_size=128,
    lr=0.00001,
    gamma=0.99,
    epsilon=1,
    min_epsilon=0.01,
    epsilon_decay=0.999,
    target_update_freq=100,
    opponent_stochasticity=0.5,
    buffer_size=10000,
    use_wandb=USE_WANDB,
)

# trainer = TrainDQN(
#     agent=agent,
#     device=DEVICE,
#     player_turn=PLAYER_TURN,
#     wrappers=wrappers,
#     batch_size=64,
#     lr=0.000001,
#     gamma=0.99,
#     epsilon=1,
#     min_epsilon=0.01,
#     epsilon_decay=0.99,
#     target_update_freq=100,
#     buffer_size=5000,
#     use_wandb=USE_WANDB,
# )

simulator = BlokusGameSimulator(
    board_size=BOARD_SIZE,
    num_players=2,
    wrappers=wrappers,
)

test_against = [Random, Greedy, Level_10, trainer.opponent_agent]


In [ ]:
trainer.collect_trajectories(max_steps=1000, pbar=True)

In [ ]:
pbar = tqdm(range(10000), desc="Training Progress")

for i in pbar:
    trainer.collect_trajectories(max_steps=64, pbar=False)
    trainer.optimize_model()
    trainer.update_epsilon()
    if trainer.step_count % 100 == 0:
        descriptions = []
        for opponent in test_against:
            agent_name = opponent.name
            log = simulator.test(
                agent1=agent,
                agent2=opponent,
                testing_agent_id=PLAYER_TURN,
                num_episodes=100
            )

            win, tie, lose = log["win_rate"], log["tie_rate"], log["lose_rate"]
            descriptions.append(f"{agent_name}: ({win:.2f},{tie:.2f},{lose:.2f}")

            if USE_WANDB:
                trainer.run.log({
                    f"{agent_name}/win_rate": win,
                    f"{agent_name}/tie_rate": tie,
                    f"{agent_name}/lose_rate": lose,
                    f"{agent_name}/diff_points": log["diff_points"]
                }, step=trainer.step_count)
        desc = f"[{len(trainer.replay_buffer)}, {i}] ε: {trainer.epsilon:.4f}|" + "|".join(descriptions)
        pbar.set_description(desc)

        q_values = agent.get_q_values(obs).cpu().numpy()
        sorted_indices = np.argsort(q_values)[::-1][:3]
        last_indices = np.argsort(q_values)[::-1][-3:]
        # Print out top 3 actions
        for idx in sorted_indices:
            action_id = obs["possible_actions"][idx]
            print(f"Action: {BlokusAction(board_size=BOARD_SIZE, action_id=action_id)}", q_values[idx].item())
        print("=============================================")

        # Print out last 3 actions
        for idx in last_indices:
            action_id = obs["possible_actions"][idx]
            print(f"Action: {BlokusAction(board_size=BOARD_SIZE, action_id=action_id)}", q_values[idx].item())
        print("=============================================")


In [ ]:
trainer.save_model()

In [ ]:
simulator = BlokusGameSimulator(
    board_size=BOARD_SIZE,
    num_players=2,
    wrappers=wrappers,
)

In [ ]:
win_rate, tie_rate, lose_rate = simulator.test(
    agent1=agent,
    agent2=MinActionsAfterSize,
    testing_agent_id=PLAYER_TURN,
    num_episodes=100,
    pbar=True
)
print(f"Win rate: {win_rate:.2f}, Tie rate: {tie_rate:.2f}, Lose rate: {lose_rate:.2f}.")